In [8]:
from data_frame.analytical.olap.cube_rollup_ops import CubeRollupOps
from data_frame.spark_utils import get_spark
from pyspark.sql import functions as F

In [9]:
spark = get_spark(app_name="Row Operations")

In [10]:

sales_cube = [
    ("North", "Electronics", "Q1", 1000),
    ("North", "Electronics", "Q2", 1500),
    ("South", "Electronics", "Q1", 2000),
    ("North", "Clothing", "Q1", 800),
    ("South", "Clothing", "Q2", 1200),
    ("East", "Electronics", "Q1", 1800)
]
df_cube = spark.createDataFrame(sales_cube, 
                               ["region", "category", "quarter", "revenue"])


## 1. CUBE Operation

In [11]:

# CUBE aggregation
cube_result = CubeRollupOps.cube_aggregation(
    df_cube, 
    ["region", "category", "quarter"],
    [F.sum("revenue").alias("total_revenue"), F.count("*").alias("num_sales")]
)
print("CUBE Result (all combinations):")
cube_result.orderBy("region", "category", "quarter").show()

CUBE Result (all combinations):
+------+-----------+-------+-------------+---------+
|region|   category|quarter|total_revenue|num_sales|
+------+-----------+-------+-------------+---------+
|  NULL|       NULL|   NULL|         8300|        6|
|  NULL|       NULL|     Q1|         5600|        4|
|  NULL|       NULL|     Q2|         2700|        2|
|  NULL|   Clothing|   NULL|         2000|        2|
|  NULL|   Clothing|     Q1|          800|        1|
|  NULL|   Clothing|     Q2|         1200|        1|
|  NULL|Electronics|   NULL|         6300|        4|
|  NULL|Electronics|     Q1|         4800|        3|
|  NULL|Electronics|     Q2|         1500|        1|
|  East|       NULL|   NULL|         1800|        1|
|  East|       NULL|     Q1|         1800|        1|
|  East|Electronics|   NULL|         1800|        1|
|  East|Electronics|     Q1|         1800|        1|
| North|       NULL|   NULL|         3300|        3|
| North|       NULL|     Q1|         1800|        2|
| North|      

## 2. ROLLUP Operation

In [12]:
rollup_result = CubeRollupOps.rollup_aggregation(
    df_cube,
    ["region", "category", "quarter"],
    [F.sum("revenue").alias("total_revenue")]
)
print("ROLLUP Result (hierarchical):")
rollup_result.orderBy("region", "category", "quarter").show()

ROLLUP Result (hierarchical):
+------+-----------+-------+-------------+
|region|   category|quarter|total_revenue|
+------+-----------+-------+-------------+
|  NULL|       NULL|   NULL|         8300|
|  East|       NULL|   NULL|         1800|
|  East|Electronics|   NULL|         1800|
|  East|Electronics|     Q1|         1800|
| North|       NULL|   NULL|         3300|
| North|   Clothing|   NULL|          800|
| North|   Clothing|     Q1|          800|
| North|Electronics|   NULL|         2500|
| North|Electronics|     Q1|         1000|
| North|Electronics|     Q2|         1500|
| South|       NULL|   NULL|         3200|
| South|   Clothing|   NULL|         1200|
| South|   Clothing|     Q2|         1200|
| South|Electronics|   NULL|         2000|
| South|Electronics|     Q1|         2000|
+------+-----------+-------+-------------+



## 3. Grouping Sets Simulation

In [13]:
# Simulate GROUPING SETS
metrics = [F.sum("revenue").alias("total_revenue")]
grouping_sets = [
    ["region", "category"],
    ["region"],
    ["category"],
    []  # Total
]

grouping_result = CubeRollupOps.grouping_sets_aggregation(
    df_cube, grouping_sets, metrics
)
print("GROUPING SETS Result:")
grouping_result.show()

AnalysisException: [NUM_COLUMNS_MISMATCH] UNION can only be performed on inputs with the same number of columns, but the first input has 3 columns and the second input has 2 columns.;
'Union false, false
:- Aggregate [region#124, category#125], [region#124, category#125, sum(revenue#127L) AS total_revenue#216L]
:  +- LogicalRDD [region#124, category#125, quarter#126, revenue#127L], false
+- Project [region#242 AS region#246, total_revenue#216L AS total_revenue#247L]
   +- Aggregate [region#242], [region#242, sum(revenue#245L) AS total_revenue#216L]
      +- LogicalRDD [region#242, category#243, quarter#244, revenue#245L], false
